# Lecture 3: Distributed Paradigms

## 1. Network Topology and Bandwidth
Why does Spark fail at parameter aggregation for Deep Learning?
In standard **Ethernet** architectures (like Hadoop/Spark clusters), communication relies on TCP/IP overhead and standard hierarchical switches.
Modern AI clusters use **InfiniBand / NVLink** which features Direct Memory Access (RDMA) and synchronized ring topologies bypassing CPU networking stacks.


## 2. Distributed Strategies: MapReduce vs Collective Communication

**The Parameter Server (Spark-Style Hub-and-Spoke):**
Let $M$ be model size (bytes), $N$ be nodes, $B$ be bandwidth. 
The central parameter server must receive $M \times N$ bytes, creating a massive bottleneck on a single machine's NIC:
$$T_{PS} = \frac{M \cdot N}{B}$$

**Ring-AllReduce (Collective Communication):**
Nodes are organized in a logical ring.
- **Scatter-reduce:** $N-1$ steps, each transferring $\frac{M}{N}$ bytes.
- **All-gather:** $N-1$ steps, each transferring $\frac{M}{N}$ bytes.

Total communication time:
$$T_{Ring} = 2 \times \frac{M(N-1)}{N \cdot B}$$

As the number of nodes $N \to \infty$, the time is bounded by:
$$T_{Ring} \approx \frac{2M}{B}$$
This algorithm scales infinitely without a central bottleneck.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, VBox, HBox, Output
import ipywidgets as widgets

def plot_communication(N_max=30000, M=10.0, B=100.0):
    # N array from 2 to N_max
    N_vals = np.linspace(2, N_max, 500)
    
    # Calculate time
    T_PS = (M * N_vals) / B
    T_Ring = 2 * (M * (N_vals - 1)) / (N_vals * B)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(N_vals, T_PS, label='Parameter Server ($T_{PS}$)', color='red', linewidth=2)
    ax.plot(N_vals, T_Ring, label='Ring-AllReduce ($T_{Ring}$)', color='blue', linewidth=2)
    
    # Limit y scale to see the Ring plateau clearly
    ax.set_ylim(0, max(2 * M / B * 3, 2))
    ax.set_xlim(2, N_max)
    
    ax.set_xlabel('Number of Nodes ($N$)')
    ax.set_ylabel('Communication Time (Seconds)')
    ax.set_title(f'Scaling Comparison (Size: {M} GB, Bandwidth: {B} GB/s)')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# Set up sliders (log-like behavior can be achieved by reading values, but here we use linear sliders for simplification per instructions)
N_slider = IntSlider(min=2, max=30000, step=100, value=1000, description='Max Nodes ($N$)')
M_slider = FloatSlider(min=0.1, max=100.0, step=0.1, value=10.0, description='Model (GB)')
B_slider = FloatSlider(min=1.0, max=400.0, step=1.0, value=100.0, description='BW (GB/s)')

# Use interactive output
out = widgets.interactive_output(plot_communication, {'N_max': N_slider, 'M': M_slider, 'B': B_slider})

display(VBox([HBox([N_slider, M_slider, B_slider]), out]))

### Exercise
Manipulate the sliders above to simulate a 30,000 GPU cluster. Why does the Parameter Server line approach vertical infinity while the Ring-AllReduce line plateaus?
Explain why modern GPU clusters are built as toruses rather than hub-and-spoke networks below:

**(Student explanation here)**